In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
from src.models import CFModel

## CF Demo

Load the M1-trained ALS model from pkl and verify that recommendations are
numerically identical to the M1 baseline (same seed=42, same user).

In [ ]:
from pathlib import Path
import time
from src.models import CFModel
from src.data import load_msd_triplets

model_path = Path('../models/als_model.pkl')

if model_path.exists():
    print('Loading pre-trained ALS from disk...')
    cf = CFModel.load('../models')
else:
    print('No pre-trained model found. Training ALS (should take ~2 min with OPENBLAS fix)...')
    df = load_msd_triplets()
    print(f'Loaded {len(df):,} interactions, {df["uid"].nunique():,} users, {df["sid"].nunique():,} songs')
    
    cf = CFModel(factors=64, regularization=0.1, iterations=20, alpha=40.0, random_state=42)
    
    t0 = time.time()
    cf.fit(df)
    elapsed = time.time() - t0
    print(f'ALS training took {elapsed:.1f}s ({elapsed/20:.2f}s per iteration)')
    
    cf.save('../models')
    print('Saved model artifacts to ../models/')

print('---')
print('Users in index:', len(cf._user_to_idx))
print('Songs in index:', len(cf._idx_to_song))

In [ ]:
# M1 baseline user -- first row of train_triplets.txt
M1_USER = 'b80344d063b5ccb3212f76538f3d9e43d87dca9e'

recs = cf.recommend(M1_USER, k=10)
print('Top-10 CF recs (should be numerically identical to M1 output):')
display(recs)

# M1 expected order:
# SOSPXWA12AB0181875  1.042114
# SOERYLG12A6701F07F  1.022091
# SOJSTYO12A8C13F200  1.015826
# SOKUTUM12A6701D9CD  0.976992
# SOOABBO12A6701DFDA  0.898471
# SOUMOMJ12A6701DFDC  0.879819
# SOVGLTY12AF72A39CD  0.879265
# SOTHABI12A58A7DACB  0.873782
# SOMYECL12A6701D9C8  0.870232
# SOGCDYR12AC961854A  0.869973

## NaN metadata fix

M1 bug: `recommend_cf()` + left-join on song_features produced NaN title/artist/genre for
17/20 top recs because the content catalog only covers ~11K of the 98K CF songs.

`recommend_with_metadata()` fills missing rows with 'Unknown' instead.

In [ ]:
# Build metadata catalog from the FULL matched set (not deduped).
# dedup_tracks() collapses 12,310 unique song_ids to 7,611 by cleaning title/artist,
# which is correct for the content k-NN index but wrong for CF metadata lookup.
# build_metadata_catalog preserves every MSD song_id.
from src.data import (
    load_spotify_tracks, load_msd_metadata, match_datasets, build_metadata_catalog
)

spotify  = load_spotify_tracks()
msd_meta = load_msd_metadata()
matched  = match_datasets(spotify, msd_meta)
meta     = build_metadata_catalog(matched)

print('Metadata covers', meta['song_id'].nunique(), 'songs (out of 98K in CF catalog)')
print('Remaining CF songs will show as Unknown.\n')

recs_with_meta = cf.recommend_with_metadata(M1_USER, k=20, metadata_df=meta)
print('Top-20 CF recs with metadata (no NaN):')
display(recs_with_meta)

In [ ]:
nan_count = (recs_with_meta['title'] == 'Unknown').sum()
print(f"'Unknown' entries: {nan_count}/20  (was NaN in M1 -- bug fixed)")

In [ ]:
import sys
sys.path.insert(0, '..')

from src.data import load_spotify_tracks, load_msd_metadata, match_datasets, dedup_tracks
from src.models import CFModel

# Reload the trained model
cf = CFModel.load('../models')

spotify = load_spotify_tracks()
msd_meta = load_msd_metadata()
matched = match_datasets(spotify, msd_meta)
matched_dedup = dedup_tracks(matched)

cf_song_ids = set(cf._idx_to_song.values())
matched_song_ids = set(matched_dedup['song_id'])
overlap = cf_song_ids & matched_song_ids

print(f'CF catalog: {len(cf_song_ids):,} songs')
print(f'Matched subset: {len(matched_song_ids):,} songs')
print(f'Overlap: {len(overlap):,} songs ({100*len(overlap)/len(cf_song_ids):.1f}%)')

example_user = list(cf._user_to_idx.keys())[100]
recs = cf.recommend(example_user, k=20)
in_matched = recs['song_id'].isin(matched_song_ids).sum()
print(f'\nExample user top-20 recs: {in_matched}/20 in matched subset')

In [ ]:
print(f'Matched (before dedup): {len(matched):,} rows, {matched["song_id"].nunique():,} unique song_ids')
print(f'Matched (after dedup):  {len(matched_dedup):,} rows, {matched_dedup["song_id"].nunique():,} unique song_ids')